# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The actionable queue is deliberately limited to the **top 50 pages** from the frozen Assignment 6 risk-severity ranking, because **Precision@50** is the ranking cut-off that was actually evaluated. The ordering is therefore model-driven; the reason codes below explain what a human should inspect and do **not** replace the ranking score.

For this playbook I use the previously held-out six-client stress-test predictions without retuning. This is intentionally conservative: the same stress test measured **Precision@50 = 0.360**, below the fixed-rule baseline of **0.480**, even though client-grouped development validation measured **0.872** versus **0.824** for the baseline. The queue is therefore a **human-review decision-support list**, not an automated recommendation engine.

### Reason codes

- `MODEL_TOP50_RISK` — the page is in the top 50 under the frozen model ranking. Every queued page receives this code.
- `LOW_CTR_FOR_POSITION` — March CTR is in the bottom quartile among pages in the same March search-position band. Assignment 5 found this signal directionally ordered and marked it **CONFIRMED**.
- `STALE_366_PLUS` — content is at least 366 days old. Assignment 5 found staleness **MIXED**, so this is a review cue/priority context only, never a standalone claim that age caused decline.

### Archetype → action mapping

| Archetype | Evidence pattern | Human-review action |
|---|---|---|
| `CTR_AND_STALE` | low CTR for position + 366+ days old | `REVIEW_CTR_AND_CONTENT_REFRESH` |
| `CTR_OPPORTUNITY` | low CTR for position, not 366+ days old | `REVIEW_SEARCH_SNIPPET_AND_INTENT` |
| `STALE_RISK` | 366+ days old, without the low-CTR flag | `REVIEW_CONTENT_FRESHNESS` |
| `MODEL_RISK_ONLY` | top-50 model risk without either previously audited action signal | `DIAGNOSE_BEFORE_EDIT` |

The final archetype is important: a high model score is enough to justify **inspection**, but not enough to invent a specific content edit. Priority tiers P1/P2/P3 simply partition ranks 1–10, 11–25, and 26–50 for reviewer workload; they are not calibrated confidence bands.

The exported queue contains March-safe features, scores, reason codes, and actions. It deliberately excludes April outcomes.

In [1]:
# SECTION 1 — build the validated top-50 human-review queue.
# Ranking order comes from the frozen Assignment 6 model.
# Reason codes are explanatory review cues, not causal prescriptions.

import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# ---------- paths + committed receipts ----------
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "baseline_split_manifest.json", "r", encoding="utf-8") as fh:
    split_manifest = json.load(fh)
with open(output_dir / "assignment6_model_benchmark.json", "r", encoding="utf-8") as fh:
    assignment6_benchmark = json.load(fh)
with open(output_dir / "assignment7_split_audit.json", "r", encoding="utf-8") as fh:
    assignment7_split = json.load(fh)
with open(output_dir / "assignment7_leakage_audit.json", "r", encoding="utf-8") as fh:
    assignment7_leakage = json.load(fh)

# ---------- warehouse access ----------
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required to rebuild the Assignment 8 queue.")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# ---------- rebuild the exact locked 2,520-page population ----------
march_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

april_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

matched_keys = (
    march_cov[march_cov["march_usable_days"] >= 20]
    .merge(april_cov, on=["client_hash_id", "content_hash_id"], how="inner")
)
matched_keys = matched_keys[
    matched_keys["april_usable_days"] >= 20
][["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("matched_keys", matched_keys)

march_exposure = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

march_exposure["exposure_tier"] = pd.cut(
    march_exposure["march_avg_impressions_per_day"],
    bins=[-float("inf"), 13.42, 58.25, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)

client_tier_counts = (
    march_exposure.groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size().unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)
eligible_clients = client_tier_counts[
    client_tier_counts.min(axis=1) >= 40
].index.tolist()

balanced_poc = (
    march_exposure[march_exposure["client_hash_id"].isin(eligible_clients)]
    .sort_values(["client_hash_id", "exposure_tier", "content_hash_id"])
    .groupby(["client_hash_id", "exposure_tier"], observed=False, group_keys=False)
    .head(40)
    .reset_index(drop=True)
)
balanced_keys = balanced_poc[
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()
con.register("balanced_keys", balanced_keys)

# ---------- March-only model features ----------
march_features = con.sql(f"""
    WITH daily AS (
        SELECT f.client_hash_id, f.content_hash_id, f.report_date,
               DATE_DIFF('day', DATE '2026-03-01', f.report_date)::DOUBLE AS day_index,
               f.gsc_impressions::DOUBLE AS impressions,
               f.gsc_clicks::DOUBLE AS clicks,
               CASE WHEN f.gsc_avg_position >= 1
                    THEN f.gsc_avg_position::DOUBLE ELSE NULL END AS valid_position
        FROM {MARCH} AS f
        INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
        WHERE f.gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
           SUM(clicks) / NULLIF(SUM(impressions), 0) AS aggregate_ctr,
           MEDIAN(valid_position) AS median_position,
           REGR_SLOPE(valid_position, day_index)
               FILTER (WHERE valid_position IS NOT NULL) AS position_slope_per_day,
           QUANTILE_CONT(valid_position, 0.75) - QUANTILE_CONT(valid_position, 0.25)
               AS position_iqr
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

age_feature = con.sql(f"""
    SELECT d.client_hash_id, d.content_hash_id,
           DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')::DOUBLE
               AS content_age_days
    FROM {DIM_CONTENT} AS d
    INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
""").df()

FINAL_FEATURES = [
    "aggregate_ctr",
    "median_position",
    "position_slope_per_day",
    "position_iqr",
    "content_age_days",
]

feature_frame = (
    march_features
    .merge(age_feature, on=["client_hash_id", "content_hash_id"], how="inner")
    .sort_values(["client_hash_id", "content_hash_id"])
    .reset_index(drop=True)
)

# ---------- April outcome is used only to fit/evaluate the already-frozen model ----------
con.register(
    "model_keys",
    feature_frame[["client_hash_id", "content_hash_id"]].drop_duplicates(),
)

march_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN model_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS april_avg_impressions_per_day
    FROM {APRIL} AS f
    INNER JOIN model_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

target_frame = march_target.merge(
    april_target,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)
target_frame["future_impression_change"] = (
    target_frame["april_avg_impressions_per_day"]
    - target_frame["march_avg_impressions_per_day"]
) / target_frame["march_avg_impressions_per_day"]
target_frame["future_decline"] = (
    target_frame["future_impression_change"] < 0
).astype(int)

modeling_frame = (
    feature_frame.merge(
        target_frame[
            [
                "client_hash_id",
                "content_hash_id",
                "future_impression_change",
                "future_decline",
            ]
        ],
        on=["client_hash_id", "content_hash_id"],
        how="inner",
    )
    .sort_values(["client_hash_id", "content_hash_id"])
    .reset_index(drop=True)
)

train_clients = set(split_manifest["train_clients"])
test_clients = set(split_manifest["test_clients"])

train_frame = (
    modeling_frame[modeling_frame["client_hash_id"].isin(train_clients)]
    .sort_values(["client_hash_id", "content_hash_id"])
    .reset_index(drop=True)
)
test_frame = (
    modeling_frame[modeling_frame["client_hash_id"].isin(test_clients)]
    .sort_values(["client_hash_id", "content_hash_id"])
    .reset_index(drop=True)
)

assert len(modeling_frame) == 2520
assert modeling_frame["client_hash_id"].nunique() == 21
assert len(train_frame) == 1800
assert train_frame["client_hash_id"].nunique() == 15
assert len(test_frame) == 720
assert test_frame["client_hash_id"].nunique() == 6
assert set(train_frame["client_hash_id"]).isdisjoint(set(test_frame["client_hash_id"]))
assert feature_frame[FINAL_FEATURES].notna().all().all()
assert assignment7_leakage["forbidden_feature_overlap"] == []

# ---------- frozen Assignment 6 models ----------
classifier = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
    class_weight=None,
    max_depth=4,
    max_features="sqrt",
    min_samples_leaf=5,
)
regressor = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
    max_depth=None,
    max_features="sqrt",
    min_samples_leaf=30,
)

X_train = train_frame[FINAL_FEATURES]
y_train_cls = train_frame["future_decline"].astype(int)
y_train_reg = train_frame["future_impression_change"].astype(float)
X_test = test_frame[FINAL_FEATURES]

classifier.fit(X_train, y_train_cls)
regressor.fit(X_train, y_train_reg)

p_decline = classifier.predict_proba(X_test)[:, 1]
predicted_future_change = regressor.predict(X_test)

rank_cfg = assignment6_benchmark["ranking"]["model"]
RANK_GAMMA = float(rank_cfg["gamma"])
RANK_LAMBDA = float(rank_cfg["lambda"])
SEVERITY_SCALE = float(rank_cfg["severity_scale_from_training_oof"])

predicted_decline_severity = np.maximum(0.0, -predicted_future_change)
severity_norm = np.clip(
    predicted_decline_severity / SEVERITY_SCALE,
    0.0,
    1.0,
)
ranking_score = np.power(
    np.clip(p_decline, 1e-9, 1.0),
    RANK_GAMMA,
) * (1.0 + RANK_LAMBDA * severity_norm)

# ---------- previously established March-only reason signals ----------
# Assignment 5 defined low CTR relative to comparable search position using
# within-position-band March CTR percentiles across the locked 2,520-page POC.
reason_frame = feature_frame.copy()
reason_frame["position_band"] = pd.cut(
    reason_frame["median_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True,
)
reason_frame["ctr_within_position_percentile"] = (
    reason_frame
    .groupby("position_band", observed=False)["aggregate_ctr"]
    .rank(method="average", pct=True)
)
reason_frame["low_ctr_for_position"] = (
    reason_frame["ctr_within_position_percentile"] <= 0.25
)
reason_frame["stale_366_plus"] = (
    reason_frame["content_age_days"] >= 366
)

scored = test_frame[
    ["client_hash_id", "content_hash_id"] + FINAL_FEATURES
].copy()
scored["p_decline"] = p_decline
scored["predicted_future_change"] = predicted_future_change
scored["predicted_decline_severity"] = predicted_decline_severity
scored["ranking_score"] = ranking_score

scored = scored.merge(
    reason_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "position_band",
            "ctr_within_position_percentile",
            "low_ctr_for_position",
            "stale_366_plus",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left",
)

scored = (
    scored.sort_values(
        ["ranking_score", "client_hash_id", "content_hash_id"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)
scored.insert(0, "rank", np.arange(1, len(scored) + 1))

# The validated ranking question is Precision@50, so the actionable playbook
# deliberately stops at 50 rather than manufacturing an unvalidated cut-off.
queue = scored.head(50).copy()

def archetype_from_row(row):
    if row["low_ctr_for_position"] and row["stale_366_plus"]:
        return "CTR_AND_STALE"
    if row["low_ctr_for_position"]:
        return "CTR_OPPORTUNITY"
    if row["stale_366_plus"]:
        return "STALE_RISK"
    return "MODEL_RISK_ONLY"

ACTION_MAP = {
    "CTR_AND_STALE": "REVIEW_CTR_AND_CONTENT_REFRESH",
    "CTR_OPPORTUNITY": "REVIEW_SEARCH_SNIPPET_AND_INTENT",
    "STALE_RISK": "REVIEW_CONTENT_FRESHNESS",
    "MODEL_RISK_ONLY": "DIAGNOSE_BEFORE_EDIT",
}

def reason_codes_from_row(row):
    codes = ["MODEL_TOP50_RISK"]
    if row["low_ctr_for_position"]:
        codes.append("LOW_CTR_FOR_POSITION")
    if row["stale_366_plus"]:
        codes.append("STALE_366_PLUS")
    return "|".join(codes)

def reason_detail_from_row(row):
    parts = ["top-50 by the frozen risk-severity ranking"]
    if row["low_ctr_for_position"]:
        parts.append("March CTR is in the bottom quartile for a similar search-position band")
    if row["stale_366_plus"]:
        parts.append("content age is at least 366 days")
    if len(parts) == 1:
        parts.append("no previously audited content-action signal is strong enough to prescribe an edit")
    return "; ".join(parts)

queue["archetype"] = queue.apply(archetype_from_row, axis=1)
queue["reason_code"] = queue.apply(reason_codes_from_row, axis=1)
queue["reason_detail"] = queue.apply(reason_detail_from_row, axis=1)
queue["action_label"] = queue["archetype"].map(ACTION_MAP)
queue["review_priority"] = pd.cut(
    queue["rank"],
    bins=[0, 10, 25, 50],
    labels=["P1", "P2", "P3"],
    include_lowest=True,
).astype(str)
queue["human_review_required"] = True
queue["automation_allowed"] = False

queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "ranking_score",
    "p_decline",
    "predicted_future_change",
    "predicted_decline_severity",
    "review_priority",
    "archetype",
    "reason_code",
    "reason_detail",
    "action_label",
    "human_review_required",
    "automation_allowed",
    "aggregate_ctr",
    "median_position",
    "position_band",
    "ctr_within_position_percentile",
    "content_age_days",
]

queue_path = output_dir / "action_playbook_queue.csv"
queue[queue_columns].to_csv(queue_path, index=False)

# ---------- evidence receipt for the paper ----------
test_truth = test_frame[
    ["client_hash_id", "content_hash_id", "future_decline"]
]
queue_eval = queue[
    ["client_hash_id", "content_hash_id", "rank"]
].merge(
    test_truth,
    on=["client_hash_id", "content_hash_id"],
    how="left",
)
observed_precision_at_50 = float(queue_eval["future_decline"].mean())

expected_stress_p50 = float(
    assignment6_benchmark["ranking"]["model"]["precision_at_50"]
)
assert np.isclose(observed_precision_at_50, expected_stress_p50)
assert np.isclose(expected_stress_p50, 0.36)

reason_counts = {
    code: int(queue["reason_code"].str.contains(code, regex=False).sum())
    for code in [
        "MODEL_TOP50_RISK",
        "LOW_CTR_FOR_POSITION",
        "STALE_366_PLUS",
    ]
}
archetype_counts = {
    str(k): int(v)
    for k, v in queue["archetype"].value_counts().sort_index().items()
}

metrics = {
    "queue_definition": {
        "source_population": "previously held-out six-client stress-test set",
        "source_pages": int(len(test_frame)),
        "queue_size": int(len(queue)),
        "validated_cutoff": "top 50 because Assignment 6/7 evaluated ranking at Precision@50",
        "ranking_formula": assignment6_benchmark["ranking"]["formula"],
        "gamma": RANK_GAMMA,
        "lambda": RANK_LAMBDA,
        "severity_scale": SEVERITY_SCALE,
    },
    "reason_code_counts": reason_counts,
    "archetype_counts": archetype_counts,
    "action_map": ACTION_MAP,
    "observed_stress_test_precision_at_50": observed_precision_at_50,
    "observed_stress_test_true_declines_in_top50": int(
        queue_eval["future_decline"].sum()
    ),
    "grouped_development_precision_at_50": float(
        {
            row["split"]: row
            for row in assignment7_split["summary"]
        }["grouped_client_cv"]["ranking_precision_at_50"]
    ),
    "guardrails": {
        "human_review_required_for_every_row": True,
        "automatic_edit_or_publish_allowed": False,
        "reason_codes_are_causal_claims": False,
        "future_outcome_written_to_queue": False,
    },
}

metrics_path = output_dir / "assignment8_action_playbook_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as fh:
    json.dump(metrics, fh, indent=2)

# ---------- compact, identifier-free notebook output ----------
archetype_summary = (
    queue.groupby(["archetype", "action_label"], as_index=False)
    .agg(
        pages=("rank", "size"),
        median_rank=("rank", "median"),
        median_p_decline=("p_decline", "median"),
        median_predicted_decline_severity=(
            "predicted_decline_severity",
            "median",
        ),
    )
    .sort_values(["pages", "archetype"], ascending=[False, True])
    .reset_index(drop=True)
)

reason_summary = pd.DataFrame(
    [{"reason_code": k, "pages": v} for k, v in reason_counts.items()]
)

print("ASSIGNMENT 8 — SECTION 1 RANKED ACTIONS")
print("Stress-test pages scored:", len(scored))
print("Human-review queue size:", len(queue))
print("Grouped-development Precision@50:", round(metrics["grouped_development_precision_at_50"], 3))
print("Six-client stress-test Precision@50:", round(observed_precision_at_50, 3))
print("Automation allowed:", queue["automation_allowed"].any())
print("\nARCHETYPE -> ACTION SUMMARY")
display(archetype_summary)
print("\nREASON-CODE COVERAGE")
display(reason_summary)
print("\nQueue written:", queue_path)
print("Metrics receipt written:", metrics_path)


ASSIGNMENT 8 — SECTION 1 RANKED ACTIONS
Stress-test pages scored: 720
Human-review queue size: 50
Grouped-development Precision@50: 0.872
Six-client stress-test Precision@50: 0.36
Automation allowed: False

ARCHETYPE -> ACTION SUMMARY


,archetype,action_label,pages,median_rank,median_p_decline,median_predicted_decline_severity
0,MODEL_RISK_ONLY,DIAGNOSE_BEFORE_EDIT,20,32.5,0.866589,0.466577
1,STALE_RISK,REVIEW_CONTENT_FRESHNESS,17,15.0,0.903015,0.484551
2,CTR_AND_STALE,REVIEW_CTR_AND_CONTENT_REFRESH,8,11.5,0.907849,0.500579
3,CTR_OPPORTUNITY,REVIEW_SEARCH_SNIPPET_AND_INTENT,5,48.0,0.867536,0.426537



REASON-CODE COVERAGE


,reason_code,pages
0,MODEL_TOP50_RISK,50
1,LOW_CTR_FOR_POSITION,13
2,STALE_366_PLUS,25



Queue written: ../outputs/action_playbook_queue.csv
Metrics receipt written: ../outputs/assignment8_action_playbook_metrics.json


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.